# Ch.7 — MLE & Loss Functions

## Core Idea

**MLE:** choose $\hat{\theta} = \arg\max_\theta \prod_i p(y_i \mid x_i;\,\theta)$.

Taking the log and negating gives the **training loss**:

| Output | Noise model | Loss |
|---|---|---|
| Continuous | $y \sim \mathcal{N}(\hat{y}, \sigma^2)$ | **MSE** |
| Binary | $y \sim \text{Bern}(\hat{p})$ | **Binary cross-entropy** |
| Multi-class | $y \sim \text{Cat}(\hat{\mathbf{p}})$ | **Categorical cross-entropy** |

## Running Example

Dataset: **California Housing** (sklearn) — 20,640 districts.
- **Regression task:** predict `MedHouseVal` (continuous) → MSE
- **Classification task:** predict `high_value = (MedHouseVal > median)` (binary) → BCE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing  = fetch_california_housing()
X, y_reg = housing.data, housing.target

threshold = np.median(y_reg)
y_clf     = (y_reg > threshold).astype(int)
print(f"Threshold: {threshold:.3f}   Positive class: {y_clf.mean()*100:.1f}%")

scaler   = StandardScaler()
X_sc     = scaler.fit_transform(X)

X_tr, X_te, yr_tr, yr_te = train_test_split(X_sc, y_reg, test_size=0.2, random_state=42)
_,    _,    yc_tr, yc_te = train_test_split(X_sc, y_clf, test_size=0.2, random_state=42)

## MSE Derived from Gaussian MLE

The Gaussian log-likelihood for one observation is:
$$\log p(y_i \mid \hat{y}_i) = -\frac{1}{2}\log(2\pi\sigma^2) - \frac{(y_i - \hat{y}_i)^2}{2\sigma^2}$$

Sum over $n$ observations, drop constants, negate → $\text{MSE} = \frac{1}{n}\sum(y_i - \hat{y}_i)^2$

In [ ]:
# ── Loss functions from scratch ───────────────────────────────────────────────
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def huber(y_true, y_pred, delta=1.0):
    r = np.abs(y_true - y_pred)
    return np.where(r <= delta, 0.5 * r**2, delta * (r - 0.5 * delta)).mean()

def bce(y_true, p_pred, eps=1e-12):
    p = np.clip(p_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

# ── Quick sanity check ────────────────────────────────────────────────────────
y_dummy = np.array([1.0, 2.0, 3.0])
p_dummy = np.array([0.9, 0.1, 0.8])
y_bin   = np.array([1,   0,   1  ])

print(f"MSE  ({y_dummy} vs {y_dummy+0.5}): {mse(y_dummy, y_dummy+0.5):.4f}")
print(f"BCE  ({y_bin} vs {p_dummy})    : {bce(y_bin, p_dummy):.4f}")

## Visualising the Losses

Plot each loss as a function of the residual (regression) or predicted probability (classification).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Regression losses vs residual ─────────────────────────────────────────────
r = np.linspace(-3, 3, 500)
delta = 1.0
axes[0].plot(r, r**2,         label='MSE  ($(r)^2$)',                     lw=2)
axes[0].plot(r, np.abs(r),    label='MAE  ($|r|$)',                       lw=2, linestyle='--')
axes[0].plot(r,
             np.where(np.abs(r) <= delta, 0.5*r**2,
                      delta*(np.abs(r)-0.5*delta)),
             label=f'Huber (δ={delta})',                                  lw=2, linestyle=':')
axes[0].set_xlabel('Residual $r = y - \\hat{y}$')
axes[0].set_ylabel('Loss')
axes[0].set_title('Regression losses')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# ── BCE vs predicted probability (for y=1) ────────────────────────────────────
p = np.linspace(0.001, 0.999, 500)
axes[1].plot(p, -np.log(p),         label='BCE (y=1): $-\\log(\\hat{p})$', lw=2)
axes[1].plot(p, (1-p)**2,           label='MSE (y=1): $(\\hat{p}-1)^2$',  lw=2, linestyle='--')
axes[1].axvline(0.99, color='gray', linestyle=':', lw=1, label='p̂=0.99 (near correct)')
axes[1].set_xlabel('Predicted probability $\\hat{p}$ (for y=1)')
axes[1].set_ylabel('Loss')
axes[1].set_title('BCE vs MSE for classification (y=1)')
axes[1].set_ylim(0, 5); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Gradient Analysis: Why MSE Fails for Classification

Near a confident correct prediction ($\hat{p}=0.99$, $y=1$), compare the gradient magnitude for BCE vs MSE.

In [ ]:
p_vals = np.array([0.01, 0.10, 0.50, 0.90, 0.99, 0.999])
y_true = 1  # true class is positive

# dL/dp̂
grad_bce = -y_true / p_vals                   # -y/p̂
grad_mse = 2 * (p_vals - y_true)              # 2(p̂ - y)

print(f"{'p̂':>8}  {'∂BCE/∂p̂':>12}  {'∂MSE/∂p̂':>12}  {'Ratio |BCE|/|MSE|':>18}")
for p, gb, gm in zip(p_vals, grad_bce, grad_mse):
    ratio = abs(gb) / (abs(gm) + 1e-12)
    print(f"{p:>8.3f}  {gb:>12.4f}  {gm:>12.4f}  {ratio:>18.1f}x")

print("\nAt p̂=0.99: BCE gradient is ~100x larger than MSE gradient")
print("→ MSE provides almost no learning signal when nearly correct")

## Empirical Demo: Logistic Regression — BCE vs MSE Loss

Train two logistic regression models: one minimising BCE (correct) and one minimising MSE (wrong choice). Compare validation AUC and calibration.

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier, SGDRegressor
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score

# ── BCE (log loss) — correct for binary classification ────────────────────────
lr_bce = LogisticRegression(max_iter=500, random_state=42)
lr_bce.fit(X_tr, yc_tr)
prob_bce = lr_bce.predict_proba(X_te)[:, 1]

# ── MSE — wrong for binary classification ─────────────────────────────────────
# SGDRegressor minimises MSE; we clip outputs to [0,1] as pseudo-probabilities
sgd_mse = SGDRegressor(loss='squared_error', max_iter=1000, random_state=42)
sgd_mse.fit(X_tr, yc_tr)
prob_mse = np.clip(sgd_mse.predict(X_te), 0, 1)

print("Classification with CORRECT loss (BCE):")
print(f"  AUC       = {roc_auc_score(yc_te, prob_bce):.4f}")
print(f"  Log-loss  = {log_loss(yc_te, prob_bce):.4f}")
print(f"  Accuracy  = {accuracy_score(yc_te, (prob_bce > 0.5).astype(int)):.4f}")

print("\nClassification with WRONG loss (MSE):")
print(f"  AUC       = {roc_auc_score(yc_te, prob_mse):.4f}")
print(f"  Log-loss  = {log_loss(yc_te, prob_mse):.4f}")
print(f"  Accuracy  = {accuracy_score(yc_te, (prob_mse > 0.5).astype(int)):.4f}")

## Calibration: Are the Probabilities Meaningful?

A calibrated model produces $\hat{p}=0.7$ when 70% of those predictions are actually positive. MSE-trained models often output values clustering near 0 and 1 or near 0.5 — poorly calibrated.

In [ ]:
from sklearn.calibration import calibration_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ── Calibration curves ────────────────────────────────────────────────────────
for label, preds, ax in [
    ('BCE (correct)', prob_bce, ax1),
    ('MSE (wrong)',   prob_mse, ax2),
]:
    frac_pos, mean_pred = calibration_curve(yc_te, preds, n_bins=10)
    ax.plot(mean_pred, frac_pos, 's-', label=label, markersize=6)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'Calibration curve — {label}')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Regression: MSE vs MAE vs Huber

MSE is sensitive to outliers (quadratic penalty). MAE is robust but non-differentiable at 0. Huber is a compromise: MSE for small residuals, linear for large ones.

In [ ]:
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

models_reg = {
    'Ridge (MSE loss)':  Ridge(alpha=1.0),
    'Huber (robust)':    HuberRegressor(epsilon=1.35, max_iter=500),
}

print(f"{'Model':<25}  {'RMSE':>8}  {'MAE':>8}")
for name, model in models_reg.items():
    model.fit(X_tr, yr_tr)
    y_pred = model.predict(X_te)
    rmse = mean_squared_error(yr_te, y_pred) ** 0.5
    mae_ = mean_absolute_error(yr_te, y_pred)
    print(f"{name:<25}  {rmse:>8.4f}  {mae_:>8.4f}")

print("\nHuber is less sensitive to the luxury-property outliers (high MedHouseVal)")

## Outlier Sensitivity: MSE vs MAE vs Huber

Inject artificial outliers into the regression target and measure how each model degrades.

In [ ]:
rng = np.random.default_rng(42)

outlier_fractions = [0, 0.01, 0.02, 0.05, 0.10]
results_ridge  = []
results_huber  = []

for frac in outlier_fractions:
    yr_tr_noisy = yr_tr.copy()
    n_outliers  = int(frac * len(yr_tr))
    if n_outliers > 0:
        idx = rng.choice(len(yr_tr), n_outliers, replace=False)
        yr_tr_noisy[idx] = rng.uniform(50, 500, size=n_outliers)  # extreme values

    ridge = Ridge(alpha=1.0).fit(X_tr, yr_tr_noisy)
    hbr   = HuberRegressor(epsilon=1.35).fit(X_tr, yr_tr_noisy)

    results_ridge.append(mean_squared_error(yr_te, ridge.predict(X_te)) ** 0.5)
    results_huber.append(mean_squared_error(yr_te, hbr.predict(X_te))   ** 0.5)

plt.figure(figsize=(8, 4))
plt.plot([f*100 for f in outlier_fractions], results_ridge, 'r-o', label='Ridge (MSE)')
plt.plot([f*100 for f in outlier_fractions], results_huber, 'b-o', label='Huber')
plt.xlabel('Outlier fraction (%)')
plt.ylabel('Test RMSE')
plt.title('MSE vs Huber — robustness to outliers')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## Exercises

1. **Label smoothing for classification.** Using a Keras or PyTorch SGD loop, train a binary classifier on `high_value` with label_smoothing=0.0 and label_smoothing=0.1. Compare validation BCE and calibration curves. Does smoothing improve calibration?

2. **Negative log-likelihood numerically.** Write a function `gaussian_nll(y_true, y_pred, sigma)` that returns the negative log-likelihood under a Gaussian. Verify that minimising it is equivalent to minimising MSE: for fixed `y_true` and `y_pred`, vary `sigma` and show that the optimal parameters $\hat{\theta}$ don't change with $\sigma$.

3. **Focal loss.** Implement focal loss: $\text{FL} = -\alpha(1-\hat{p})^\gamma \log(\hat{p})$ for $y=1$. Plot it alongside BCE for $\gamma \in \{0, 1, 2, 5\}$. What does increasing $\gamma$ do to the gradient near $\hat{p}=0.99$? When is this useful?

In [ ]:
# Exercise 1 — Label smoothing
# TODO: your solution here

In [ ]:
# Exercise 2 — Gaussian NLL
# TODO: your solution here

In [ ]:
# Exercise 3 — Focal loss
# TODO: your solution here